In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

In [2]:
x = torch.randn(3, 4)
print(x)
print(x.shape)
print(x.sum())

tensor([[-1.4667,  1.0504, -1.6359,  1.7183],
        [-0.2814,  0.5682, -2.5862,  0.1405],
        [-1.0482,  1.3502,  0.8515,  0.4441]])
torch.Size([3, 4])
tensor(-0.8952)


In [3]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
print(x.grad)
y = x[0]**2 + x[1]**3
y.backward()
print(x.grad)

None
tensor([ 4., 27.])


$$y = {x_0}^2 + {x_1}^3$$

$$\frac{\partial y}{\partial x_0} = 2{x_0} = 2(2)=4$$

$$\frac{\partial y}{\partial x_1} = 3{x_1}^2 = 3(3)^2 = 27$$


## Logistic Regression

In [4]:
class LogisticRegression:
    def __init__(self, lr=0.1, epochs=1000):
        # store hyperparameters
        self.lr = lr
        self.epochs = epochs
    
    def fit(self, X, y):
        
        X_train_t = torch.from_numpy(X).float()
        y_train_t = torch.from_numpy(y).float().reshape(-1, 1)

        #init w and b
        self.W = torch.randn(2, 1, requires_grad=True)
        self.b = torch.zeros(1, requires_grad=True)

        # training loop
        self.X = X_train_t
        self.Y = y_train_t
        self.L=[]


        for epoch in range(self.epochs):
    
            #find z for each x_train val
            z = self.X @ self.W + self.b 
            
            #find sigmoid prop
            y_pred = torch.sigmoid(z) 
            
            #find loss
            loss = self.logLoss(self.Y, y_pred)
            self.L.append(loss.item())
        
            # Use autograd to compute the backward pass.
            loss.backward()
        
            #update
            with torch.no_grad():
                self.W -= self.lr * self.W.grad
                self.b -= self.lr * self.b.grad
                     
                self.W.grad.zero_()
                self.b.grad.zero_()
    
    def predict_proba(self, X):
        X = torch.from_numpy(X).float()
        # return probabilities
        with torch.no_grad():
            return (torch.sigmoid(X @ self.W + self.b))
        
    
    def predict(self, X):
        # return 0/1 predictions
        return (self.predict_proba(X)>=0.5).int()
    
    def score(self, X, y):
        y = torch.from_numpy(y).float().reshape(-1, 1)
        # return accuracy
        return torch.mean((self.predict(X) == y).float())

    def loss(self):
        # return accuracy
        try:
            return self.L
        except AttributeError:
            print("Model not trained")
    
    #logLoss
    def logLoss(self,y_train_t, y_pred):
        y_pred = torch.clamp(y_pred, 1e-15, 1-1e-15)
        loss = -torch.sum(y_train_t*torch.log(y_pred) + (1-y_train_t)*torch.log(1-y_pred)) / y_train_t.shape[0]
        return loss

## Neural Net

In [5]:
class NeuralNet():
    '''
    -Input layer -> Hidden layer (RELU) -> 1 Output layer (sigmoid)

    -Binary cross-entropy loss
    
    -Backprop + gradient descent
    '''
    def __init__(self,lr=0.1,neurons=8, epochs = 100):
        self.lr = lr
        self.hidden_neurons = neurons
        self.epochs = epochs
        
    def sigmoid(self,z):
        return 1/(1+np.exp(-z))

    def forward(self,X):
        z1 = X @ self.W1 + self.b1
        a1 = z1.copy()
        a1[a1 < 0] = 0  # ReLU
        
        z2 = a1 @ self.W2 + self.b2
        a2 = self.sigmoid(z2)
        
        return z1, a1, z2, a2
    
    def backward (self,z1,a1,z2,a2):
        # Output layer
        dz2 = a2 - self.y_train
        dW2 = a1.T @ dz2 / len(self.y_train)
        db2 = np.sum(dz2, axis=0) / len(self.y_train)
        
        # Hidden layer
        da1 = dz2 @ self.W2.T
        dz1 = da1 * (z1 > 0)
        dW1 = self.X_train.T @ dz1 / len(self.y_train)
        db1 = np.sum(dz1, axis=0) / len(self.y_train)
    
        return dz2,dW2, db2, da1, dz1, dW1, db1
    
    def fit(self,X,y):
        n,d=X.shape
        self.X_train = X
        self.y_train = y.reshape(-1, 1)
        self.W1 = np.random.randn(d, self.hidden_neurons)*0.01
        self.b1 = np.random.randn(self.hidden_neurons,)*0.01
        self.W2 = np.random.randn(self.hidden_neurons,1)*0.01
        self.b2 = np.random.randn(1,)*0.01

        self.losses = []
  

        for _ in range(self.epochs):
            z1,a1,z2,a2 = self.forward(self.X_train)
            # Track loss
            loss = -np.mean(self.y_train * np.log(a2 + 1e-15) + (1 - self.y_train) * np.log(1 - a2 + 1e-15))
            self.losses.append(loss)
            dz2,dW2, db2, da1, dz1, dW1, db1 = self.backward(z1,a1,z2,a2)
            #update weights and biases
            self.W1 = self.W1 - self.lr*dW1
            self.W2 = self.W2 - self.lr*dW2
            self.b1 = self.b1 - self.lr*db1
            self.b2 = self.b2 - self.lr*db2
        
    
    def predict_proba(self,X):
        _,_,_,out = self.forward(X)
        return out

    def predict(self, X):
        # return 0/1 predictions
        return (self.predict_proba(X)>=0.5).astype(int)

    def score(self,X,y):
        return np.mean(self.predict(X).flatten() == y)

In [6]:
import torch.nn as nn

class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        # define layers
        self.input_Layer = nn.Linear(2,8)
        self.hidden_Layer = nn.Linear(8,1)
        
    def forward(self, x):
        # define forward pass
        x = self.input_Layer(x)
        x = torch.relu(x)
        x = self.hidden_Layer(x)
        x = torch.sigmoid(x)
        return x
    
    def fit(self,x,y,lr=0.1,n_epochs=1000):
        X = torch.from_numpy(x).float()
        y = torch.from_numpy(y).float().reshape(-1,1)
        
        criterion = nn.BCELoss()
        optimizer = torch.optim.SGD(self.parameters(), lr=lr)

        for epoch in range(n_epochs):
            
            y_pred = self(X) #forward pass
            loss = criterion(y_pred,y) #compute loss
        
            #get gradients
            optimizer.zero_grad()
            loss.backward()
        
            #update weights and bias
            optimizer.step()
            
            if epoch % 100 == 0:
                print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

    def predict(self,x):
        X = torch.from_numpy(x).float()
        y_pred = self(X)
        y_pred_class = (y_pred >= 0.5).int()
        return y_pred_class

    def score(self,X,y):
        y = torch.from_numpy(y).float().reshape(-1,1)
        accuracy = (self.predict(X) == y).float().mean()
        return accuracy

## TESTING

In [7]:
np.random.seed(42)
X_class0 = np.random.randn(100, 2) + np.array([0, 0])
X_class1 = np.random.randn(100, 2) + np.array([1.5, 1.5])  # closer now

X = np.vstack([X_class0, X_class1])
y = np.array([0]*100 + [1]*100)

idx = np.random.permutation(200)
X, y = X[idx], y[idx]

X_train, X_test = X[:160], X[160:]
y_train, y_test = y[:160], y[160:]


X_train_t = torch.from_numpy(X).float()
y_train_t = torch.from_numpy(y).float().reshape(-1, 1)

### Logistic Regression

In [8]:
model = LogisticRegression(lr=0.1, epochs=1000)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

tensor(0.8750)


### NN

In [9]:
model=NeuralNet()
model.fit(X_train, y_train)
model.score(X_test,y_test)

Epoch 0, Loss: 0.6393
Epoch 100, Loss: 0.2890
Epoch 200, Loss: 0.2641
Epoch 300, Loss: 0.2599
Epoch 400, Loss: 0.2583
Epoch 500, Loss: 0.2572
Epoch 600, Loss: 0.2565
Epoch 700, Loss: 0.2560
Epoch 800, Loss: 0.2555
Epoch 900, Loss: 0.2551


tensor(0.8500)